# ELM-INR: Extreme Learning Machine for Implicit Neural Representations

Implementation of the core ELM-INR method from *"Escaping Spectral Bias without
Backpropagation: Fast Implicit Neural Representations with Extreme Learning Machines"*.

**Key idea:** Represent a grayscale image as a continuous function $\hat{f}(x,y)$ using
multiple **local Extreme Learning Machines** (ELMs) trained independently in closed form
(no backpropagation), blended together with a **partition-of-unity** (PoU).

$$\hat{f}(x,y) = \sum_i \phi_i(x,y)\, \hat{f}_i(x,y)$$

where $\phi_i$ are smooth window functions satisfying $\sum_i \phi_i = 1$,
and each $\hat{f}_i$ is a local ELM:

$$\hat{f}_i(x) = H_i(x)\, \alpha_i, \qquad H_i(x)_j = \sigma(w_{ij}^\top z(x) + b_{ij})$$

with random frozen weights $w, b$ and closed-form output weights $\alpha_i$.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Added {project_root} to Python path")

In [ ]:
import math
import time
from dataclasses import dataclass
from typing import Optional, List, Literal

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['image.cmap'] = 'gray'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}  |  PyTorch {torch.__version__}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 2. ELM-INR Implementation

The full `ELMINR2D` class with:
- Coordinate generation in $[-1,1]^2$
- Overlapping rectangular subdomains on a regular 2D grid
- Partition-of-unity (PoU) windows (Hann / cosine)
- Random Fourier Features (RFF) for input encoding
- Per-subdomain ELM with closed-form ridge regression
- Full-image reconstruction via PoU blending

In [ ]:
@dataclass
class ELMINRConfig:
    """Configuration for the ELM-INR model."""
    # Subdomain grid
    grid_h: int = 8                  # number of subdomains along height
    grid_w: int = 8                  # number of subdomains along width
    overlap: float = 0.25            # fractional overlap between neighbors (0..1)
    expand_dim: int = 512                # number of extra pixels to expand each subdomain (for better blending)

    # Local ELM
    hidden_dim: int = 512            # hidden units per local ELM
    activation: Literal['relu', 'tanh', 'sigmoid'] = 'relu'

    # Input encoding
    use_rff: bool = True             # use Random Fourier Features
    rff_dim: int = 128               # number of RFF frequencies (output is 2*rff_dim)
    rff_sigma: float = 10.0          # std of Gaussian frequency matrix B

    # Solver
    ridge_lambda: float = 1e-4       # Tikhonov regularization

    # Window type for partition of unity
    window: Literal['hann', 'triangular', 'gaussian'] = 'hann'

    # Random seed for reproducibility
    seed: int = 42

In [ ]:
class ELMINR2D:
    """
    ELM-INR for 2D grayscale images.

    Represents an image I(x,y) as a sum of local ELMs blended via
    partition-of-unity windows:

        f_hat(x,y) = sum_i  phi_i(x,y) * f_hat_i(x,y)

    where phi_i are smooth windows (sum to 1) and each f_hat_i is an ELM
    with random frozen hidden layer(s) and closed-form output weights.

    Architecture per subdomain
    --------------------------
    Single-layer (expand_dim=0):
        H = σ(z @ W_h + b_h)                     [N, hidden_dim]

    Two-layer (expand_dim > 0):
        H1 = σ(z @ W_h1 + b_h1)                  [N, expand_dim]
        H  = σ(H1 @ W_h2 + b_h2)                 [N, hidden_dim]

    W_h, b_h (or W_h1/b_h1/W_h2/b_h2) are ALL frozen random weights
    regenerable from the seed — they add zero storage cost.
    Only the output weights α_i  [hidden_dim, 1] are stored.
    """

    def __init__(self, cfg: ELMINRConfig, device: torch.device = torch.device('cpu')):
        self.cfg = cfg
        self.device = device
        self.fitted = False

        # Set random seed for reproducible random weights
        self.rng = torch.Generator(device=device)
        self.rng.manual_seed(cfg.seed)

        # RFF frequency matrix B: shape [2, rff_dim]
        # (or [input_dim, rff_dim] — here input_dim=2)
        if cfg.use_rff:
            self.B = torch.randn(2, cfg.rff_dim, generator=self.rng,
                                 device=device) * cfg.rff_sigma
            self.input_dim = 2 * cfg.rff_dim  # cos + sin
        else:
            self.B = None
            self.input_dim = 2  # raw (x, y)

        # Per-subdomain random hidden weights: will be created in _init_subdomains
        self.subdomains: List[dict] = []
        self.alphas: List[torch.Tensor] = []

    # ─── Coordinate generation ────────────────────────────────────────────

    @staticmethod
    def make_coords(H: int, W: int, device: torch.device) -> torch.Tensor:
        """
        Build normalized 2D coordinates in [-1, 1] x [-1, 1].

        Returns:
            coords: [H*W, 2] tensor with (y, x) normalized coordinates.
        """
        ys = torch.linspace(-1, 1, H, device=device)
        xs = torch.linspace(-1, 1, W, device=device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
        coords = torch.stack([grid_y.flatten(), grid_x.flatten()], dim=-1)  # [N, 2]
        return coords

    # ─── Input feature encoding ──────────────────────────────────────────

    def encode(self, coords: torch.Tensor) -> torch.Tensor:
        """
        Encode coordinates into input features.

        If use_rff: z(x) = [cos(2*pi*B^T x), sin(2*pi*B^T x)]   (2*rff_dim)
        Otherwise:  z(x) = x                                      (2)

        Args:
            coords: [N, 2]
        Returns:
            features: [N, input_dim]
        """
        if self.cfg.use_rff:
            # coords @ B  ->  [N, rff_dim]
            proj = 2.0 * math.pi * (coords @ self.B)
            return torch.cat([torch.cos(proj), torch.sin(proj)], dim=-1)
        else:
            return coords

    # ─── Subdomain generation ────────────────────────────────────────────

    def _init_subdomains(self):
        """
        Create a regular 2D grid of overlapping rectangular subdomains in [-1,1]^2.

        Each subdomain is defined by:
          - center:    (cy, cx)
          - half_size: (hy, hx)  — half-extent including overlap

        With overlap ratio r, the base cell width is 2/grid_w and the
        support half-width is (1+r) * base_half.
        """
        cfg = self.cfg
        base_hy = 1.0 / cfg.grid_h  # half the non-overlapping cell height
        base_hx = 1.0 / cfg.grid_w

        # Expanded half-sizes with overlap
        hy = base_hy * (1.0 + cfg.overlap)
        hx = base_hx * (1.0 + cfg.overlap)

        self.subdomains = []
        for i in range(cfg.grid_h):
            cy = -1.0 + base_hy + 2.0 * base_hy * i  # center y
            for j in range(cfg.grid_w):
                cx = -1.0 + base_hx + 2.0 * base_hx * j  # center x

                if cfg.expand_dim > 0:
                    # Two-layer: input_dim → expand_dim → hidden_dim
                    # Both layers are frozen and regenerable from seed — zero storage cost.
                    std1 = 1.0 / math.sqrt(self.input_dim)
                    W_h1 = torch.randn(self.input_dim, cfg.expand_dim,
                                       generator=self.rng, device=self.device) * std1
                    b_h1 = torch.rand(cfg.expand_dim,
                                      generator=self.rng, device=self.device)  # U[0,1]

                    std2 = 1.0 / math.sqrt(cfg.expand_dim)
                    W_h2 = torch.randn(cfg.expand_dim, cfg.hidden_dim,
                                       generator=self.rng, device=self.device) * std2
                    b_h2 = torch.rand(cfg.hidden_dim,
                                      generator=self.rng, device=self.device)  # U[0,1]

                    self.subdomains.append({
                        'center': (cy, cx),
                        'half_size': (hy, hx),
                        'W_h1': W_h1,  # [input_dim, expand_dim]  — frozen
                        'b_h1': b_h1,  # [expand_dim]             — frozen
                        'W_h2': W_h2,  # [expand_dim, hidden_dim] — frozen
                        'b_h2': b_h2,  # [hidden_dim]             — frozen
                    })
                else:
                    # Single hidden layer: input_dim → hidden_dim
                    # FIX: use U[0,1] bias instead of N(0,1).
                    # With N(0,1) bias, ~50% of ReLU units are dead for the narrow
                    # coordinate range inside small subdomains (16x16 grid).
                    # Dead units → entire H columns are zero → H^T H rank-deficient
                    # → NaN in the solve even with ridge regularisation.
                    # U[0,1] guarantees all biases are positive so ReLU fires
                    # for any non-negative pre-activation, keeping H full-rank.
                    std = 1.0 / math.sqrt(self.input_dim)
                    W_h = torch.randn(self.input_dim, cfg.hidden_dim,
                                      generator=self.rng, device=self.device) * std
                    b_h = torch.rand(cfg.hidden_dim,
                                     generator=self.rng, device=self.device)  # U[0,1]

                    self.subdomains.append({
                        'center': (cy, cx),
                        'half_size': (hy, hx),
                        'W_h': W_h,   # [input_dim, hidden_dim] — frozen
                        'b_h': b_h,   # [hidden_dim]            — frozen
                    })

    # ─── Window (partition of unity) ─────────────────────────────────────

    def _window_1d(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluate 1D unnormalized window on [-1, 1] (zero outside).

        Args:
            t: values in [-1, 1], where 0 = center of subdomain
        Returns:
            weights >= 0, strongest at 0, zero at |t| >= 1
        """
        t = t.clamp(-1, 1)
        if self.cfg.window == 'hann':
            # Hann (raised cosine): 0.5*(1 + cos(pi*t)),  zero at |t|=1
            return 0.5 * (1.0 + torch.cos(math.pi * t))
        elif self.cfg.window == 'triangular':
            return (1.0 - t.abs())
        elif self.cfg.window == 'gaussian':
            # Truncated Gaussian (sigma=0.4)
            return torch.exp(-0.5 * (t / 0.4) ** 2)
        else:
            raise ValueError(f"Unknown window type: {self.cfg.window}")

    def _eval_window(self, coords: torch.Tensor, sub: dict) -> torch.Tensor:
        """
        Evaluate the separable 2D window phi_i(x,y) for one subdomain.

        The window is separable: phi(y,x) = w((y-cy)/hy) * w((x-cx)/hx)

        Args:
            coords: [N, 2]  (y, x)
            sub: subdomain dict with 'center' and 'half_size'
        Returns:
            phi: [N]  non-negative window weights (unnormalized)
        """
        cy, cx = sub['center']
        hy, hx = sub['half_size']

        # Normalized distance from center: in [-1, 1] inside support
        ty = (coords[:, 0] - cy) / hy
        tx = (coords[:, 1] - cx) / hx

        # Zero outside support
        inside = (ty.abs() <= 1.0) & (tx.abs() <= 1.0)
        phi = torch.zeros(coords.shape[0], device=coords.device)
        if inside.any():
            phi[inside] = self._window_1d(ty[inside]) * self._window_1d(tx[inside])
        return phi

    # ─── Hidden-layer forward pass ───────────────────────────────────────

    def _activation(self, x: torch.Tensor) -> torch.Tensor:
        if self.cfg.activation == 'relu':
            return F.relu(x)
        elif self.cfg.activation == 'tanh':
            return torch.tanh(x)
        elif self.cfg.activation == 'sigmoid':
            return torch.sigmoid(x)
        else:
            raise ValueError(f"Unknown activation: {self.cfg.activation}")

    def _hidden_matrix(self, features: torch.Tensor, sub: dict) -> torch.Tensor:
        """
        Compute hidden-layer activations H for a subdomain's ELM.

        Single-layer (expand_dim=0):
            H = sigma(z @ W_h + b_h)             [N, hidden_dim]

        Two-layer (expand_dim > 0):
            H1 = sigma(z @ W_h1 + b_h1)          [N, expand_dim]
            H  = sigma(H1 @ W_h2 + b_h2)         [N, hidden_dim]

        Args:
            features: [N_sub, input_dim]  encoded coordinates within subdomain
            sub: subdomain dict with frozen weight tensors
        Returns:
            H: [N_sub, hidden_dim]
        """
        if 'W_h1' in sub:
            h1 = self._activation(features @ sub['W_h1'] + sub['b_h1'])
            return self._activation(h1 @ sub['W_h2'] + sub['b_h2'])
        return self._activation(features @ sub['W_h'] + sub['b_h'])

    # ─── Closed-form fitting (numerically stable ridge regression) ────────

    def _solve_alpha(self, H: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        """
        Solve for output weights alpha via the augmented least-squares form
        of ridge regression.

        FIX: the original Cholesky-on-normal-equations approach (H^T H + λI)
        has two failure modes that produce NaN on fine grids (e.g. 16x16):

          1. ReLU dead neurons → entire H columns == 0 → H^T H rank-deficient.
             Even with ridge λ, Cholesky pivots can collapse to ≈λ, and the
             solve produces ±Inf which propagates as NaN.

          2. Corner subdomains with n_pts < hidden_dim → H^T H structurally
             rank-deficient regardless of λ.

          3. The lstsq fallback on CUDA does NOT raise RuntimeError for
             rank-deficient input — it silently returns NaN in .solution.

        The augmented system avoids forming H^T H entirely:

            [H          ]         [Y]
            [sqrt(λ) * I]  alpha = [0]

        torch.linalg.lstsq on this system gives the minimum-norm ridge
        solution while preserving the condition number of H (rather than
        squaring it as H^T H does).  A nan_to_num guard handles any
        residual CUDA edge cases.

        Args:
            H: [N_sub, m]  hidden matrix
            Y: [N_sub, 1]  target values
        Returns:
            alpha: [m, 1]
        """
        m = H.shape[1]
        sqrt_lam = math.sqrt(self.cfg.ridge_lambda)

        # Stack regularisation rows below the data rows
        H_aug = torch.cat(
            [H, sqrt_lam * torch.eye(m, device=H.device, dtype=H.dtype)], dim=0
        )
        Y_aug = torch.cat(
            [Y, torch.zeros(m, 1, device=Y.device, dtype=Y.dtype)], dim=0
        )

        alpha = torch.linalg.lstsq(H_aug, Y_aug).solution  # [m, 1]

        # Guard against any residual NaN/Inf (CUDA lstsq edge case)
        return torch.nan_to_num(alpha, nan=0.0, posinf=0.0, neginf=0.0)

    # ─── Fit ─────────────────────────────────────────────────────────────

    def fit(self, image: torch.Tensor, verbose: bool = True) -> 'ELMINR2D':
        """
        Fit the ELM-INR model to a grayscale image.

        1. Build coordinate grid and flatten
        2. Create overlapping subdomains
        3. For each subdomain:
           a. Select points inside its support
           b. Encode coordinates -> features
           c. Compute hidden matrix H
           d. Solve alpha in closed form

        Args:
            image: [H, W] grayscale image tensor
            verbose: print progress
        Returns:
            self
        """
        assert image.ndim == 2, f"Expected 2D image, got shape {image.shape}"
        self.H, self.W = image.shape

        t0 = time.time()

        # Step 1: coordinates and targets
        coords = self.make_coords(self.H, self.W, self.device)   # [N, 2]
        Y = image.to(self.device).flatten().unsqueeze(-1)         # [N, 1]

        # Step 2: encode all coordinates once (shared RFF matrix B)
        features = self.encode(coords)  # [N, input_dim]

        # Step 3: initialise subdomains (creates random frozen weights)
        self._init_subdomains()
        n_sub = len(self.subdomains)
        if verbose:
            expand_str = (f" → expand={self.cfg.expand_dim} → "
                          if self.cfg.expand_dim > 0 else " → ")
            arch_str = f"{self.input_dim}{expand_str}{self.cfg.hidden_dim}"
            print(f"ELM-INR: {self.cfg.grid_h}x{self.cfg.grid_w} = {n_sub} subdomains, "
                  f"arch=[{arch_str}], overlap={self.cfg.overlap}")
            print(f"  Image: {self.H}x{self.W} = {self.H*self.W:,} pixels")
            print(f"  Input features: {self.input_dim}  "
                  f"({'RFF sigma=' + str(self.cfg.rff_sigma) if self.cfg.use_rff else 'raw coords'})")

        # Step 4: fit each subdomain independently
        self.alphas = []
        total_points = 0
        for sub in self.subdomains:
            # Find points inside this subdomain's support
            cy, cx = sub['center']
            hy, hx = sub['half_size']
            mask = ((coords[:, 0] - cy).abs() <= hy) & \
                   ((coords[:, 1] - cx).abs() <= hx)

            n_pts = mask.sum().item()
            total_points += n_pts

            if n_pts == 0:
                # Empty subdomain — store zero alpha
                self.alphas.append(torch.zeros(self.cfg.hidden_dim, 1,
                                              device=self.device))
                continue

            # Features and targets for this subdomain
            feat_sub = features[mask]  # [n_pts, input_dim]
            Y_sub = Y[mask]            # [n_pts, 1]

            # Hidden matrix
            H_sub = self._hidden_matrix(feat_sub, sub)  # [n_pts, hidden_dim]

            # Closed-form solve
            alpha = self._solve_alpha(H_sub, Y_sub)  # [hidden_dim, 1]
            self.alphas.append(alpha)

        self.fitted = True
        elapsed = time.time() - t0

        if verbose:
            avg_pts = total_points / n_sub
            print(f"  Avg points/subdomain: {avg_pts:,.0f}")
            print(f"  Fit completed in {elapsed:.2f}s (no backpropagation)")

        return self

    # ─── Predict ─────────────────────────────────────────────────────────

    def predict(self, coords: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Evaluate the ELM-INR at arbitrary 2D coordinates.

        For each query point:
          1. Find all subdomains whose support contains it
          2. Evaluate local ELM predictions f_hat_i
          3. Evaluate window weights phi_i
          4. Normalize windows (partition of unity)
          5. Blend:  f_hat = sum_i phi_i * f_hat_i

        Args:
            coords: [N, 2] query points, or None to use the training grid
        Returns:
            values: [N] predicted pixel intensities
        """
        assert self.fitted, "Call fit() first"

        if coords is None:
            coords = self.make_coords(self.H, self.W, self.device)

        N = coords.shape[0]
        features = self.encode(coords)  # [N, input_dim]

        # Accumulate weighted predictions and total window weights
        weighted_sum = torch.zeros(N, device=self.device)
        weight_sum = torch.zeros(N, device=self.device)

        for sub, alpha in zip(self.subdomains, self.alphas):
            # Evaluate window for all query points
            phi = self._eval_window(coords, sub)  # [N]
            active = phi > 0

            if not active.any():
                continue

            # Local ELM prediction for active points
            H_k = self._hidden_matrix(features[active], sub)  # [n_active, hidden_dim]
            pred_k = (H_k @ alpha).squeeze(-1)                 # [n_active]

            # Accumulate
            weighted_sum[active] += phi[active] * pred_k
            weight_sum[active] += phi[active]

        # Normalize by total window weight  (PoU: sum_i phi_i = 1)
        safe_weight = weight_sum.clamp(min=1e-10)
        return weighted_sum / safe_weight

    # ─── Reconstruct image ───────────────────────────────────────────────

    def reconstruct_image(self) -> torch.Tensor:
        """
        Predict on the full training grid and reshape to H x W.

        Returns:
            recon: [H, W] tensor
        """
        vals = self.predict()
        return vals.reshape(self.H, self.W)

    # ─── Storage analysis ────────────────────────────────────────────────

    def storage_bytes(self, dtype_bytes: int = 4) -> dict:
        """
        Estimate the storage required for the ELM-INR representation.

        Stored per subdomain: alpha_i  [hidden_dim, 1]
        Shared: RFF matrix B [2, rff_dim]
        All hidden layer weights (W_h, b_h / W_h1, b_h1, W_h2, b_h2) are
        regenerable from the seed — only the seed needs to be stored.

        Returns dict with byte counts.
        """
        n_sub = len(self.subdomains)
        m = self.cfg.hidden_dim

        # Alpha weights: sum actual sizes (adaptive subdomains may have smaller hidden_dim)
        if self.alphas:
            alpha_bytes = sum(a.numel() for a in self.alphas) * dtype_bytes
        else:
            alpha_bytes = n_sub * m * 1 * dtype_bytes

        # RFF matrix B (shared, must be stored or regenerated from seed)
        if self.cfg.use_rff:
            b_matrix_bytes = 2 * self.cfg.rff_dim * dtype_bytes
        else:
            b_matrix_bytes = 0

        # Hidden weights: random and regenerable from seed (only seed needed)
        if self.cfg.expand_dim > 0:
            # Two-layer: W_h1 [input_dim×expand_dim] + b_h1 [expand_dim]
            #            W_h2 [expand_dim×hidden_dim] + b_h2 [hidden_dim]
            wh_bytes = n_sub * (
                self.input_dim * self.cfg.expand_dim
                + self.cfg.expand_dim
                + self.cfg.expand_dim * m
                + m
            ) * dtype_bytes
        else:
            wh_bytes = n_sub * self.input_dim * m * dtype_bytes
        bh_bytes = 0 if self.cfg.expand_dim > 0 else n_sub * m * dtype_bytes

        # Metadata: grid dims, overlap, seed, etc. (negligible)
        meta_bytes = 64

        return {
            'alpha_bytes': alpha_bytes,
            'rff_B_bytes': b_matrix_bytes,
            'W_h_bytes (regenerable)': wh_bytes,
            'b_h_bytes (regenerable)': bh_bytes,
            'meta_bytes': meta_bytes,
            'total_stored': alpha_bytes + b_matrix_bytes + meta_bytes,

            'total_if_all_stored': alpha_bytes + b_matrix_bytes + wh_bytes + bh_bytes + meta_bytes,
        }

### Quality metrics

In [ ]:
from skimage.metrics import structural_similarity as _skimage_ssim
def compute_psnr(pred: np.ndarray, tgt: np.ndarray) -> float:
    dr = float(tgt.max() - tgt.min())
    mse = float(np.mean((np.clip(pred, tgt.min(), tgt.max()) - tgt) ** 2))
    return 10 * math.log10(dr ** 2 / (mse + 1e-10))

def compute_ssim(pred: np.ndarray, tgt: np.ndarray) -> float:
    dr = float(tgt.max() - tgt.min())
    return float(_skimage_ssim(tgt, np.clip(pred, tgt.min(), tgt.max()), data_range=dr))

print("Metric helpers ready")

## 3. Load Dataset and Extract Frames

Uses the same `ProjectionSliceDataset` pipeline as the GSplat notebook.

In [ ]:
from inct.dataset_slices import ProjectionSliceDataset

DATA_PATH = Path('/myhome/data/sdate/shared/compression_paper/file_1_extracted')
NUM_PROJECTIONS = 10
TARGET_SIZE = (2560 // 2, 2160 // 2)  # (1280, 1080) — manageable for ELM fitting

dataset = ProjectionSliceDataset(
    folder_path=DATA_PATH,
    num_projections=NUM_PROJECTIONS,
    target_size=TARGET_SIZE,
    normalize_values=True,
    verbose=True,
    cache_volume=True,
    use_attenuation=False,
)

volume = dataset.get_full_volume()  # (H, W, D)
H, W = volume.shape[:2]
print(f"\nVolume shape: {volume.shape}")
print(f"Image size: {H} x {W}")

# Extract and per-frame normalise
frames = []
for i in range(NUM_PROJECTIONS):
    img = volume[:, :, i].clone()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    frames.append(img)

# Preview
fig, axes = plt.subplots(1, min(5, len(frames)), figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(frames[i].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Frame {i}')
    ax.axis('off')
plt.suptitle(f'{len(frames)} frames @ {H}x{W}', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Fit ELM-INR to a Single Frame

Select one frame and fit an ELM-INR representation.
No backpropagation — every local ELM is solved in closed form.

In [ ]:
# Select frame to fit
FRAME_IDX = 0
image = frames[FRAME_IDX].to(device)
# image = (frames[FRAME_IDX] - frames[FRAME_IDX + 1]).to(device)
# image -= image.min()  # shift to zero
# image /= image.max()  # scale to [0, 1]
print(f"Fitting frame {FRAME_IDX}: shape {image.shape}, "
      f"range [{image.min():.4f}, {image.max():.4f}]")

# ── Configure and fit ──────────────────────────────────────────────────
cfg = ELMINRConfig(
    grid_h=32,
    grid_w=32,
    overlap=0.25,
    hidden_dim=1024,        # alpha storage: 16×16×128×4 = 1 MB
    expand_dim=0,       # wide intermediate layer — free (regenerable from seed)
    activation='relu',
    use_rff=True,
    rff_dim=256,
    rff_sigma=5.0,
    ridge_lambda=1e-6,
    window='gaussian',
    seed=42,
)

model = ELMINR2D(cfg, device=device)
model.fit(image, verbose=True)
torch.cuda.empty_cache()


## 5. Reconstruction and Quality

In [ ]:
# ── Reconstruct ───────────────────────────────────────────────────────
t0 = time.time()
recon = model.reconstruct_image()
recon_time = time.time() - t0
print(f"Reconstruction time: {recon_time:.2f}s")

# Clip to valid range
recon_np = recon.clamp(0, 1).cpu().numpy()
orig_np = image.cpu().numpy()

# ── Quality metrics ───────────────────────────────────────────────────
mse_val = np.mean((orig_np - recon_np) ** 2)
psnr_val = compute_psnr(orig_np, recon_np)
ssim_val = compute_ssim(orig_np, recon_np)

print(f"\nQuality:")
print(f"  MSE  = {mse_val:.6f}")
print(f"  PSNR = {psnr_val:.2f} dB")
print(f"  SSIM = {ssim_val:.4f}")

# ── Visualise ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(orig_np, cmap='gray', vmin=orig_np.min(), vmax=orig_np.max())
axes[0].set_title(f'Original (frame {FRAME_IDX})', fontsize=12)
axes[0].axis('off')

axes[1].imshow(recon_np, cmap='gray', vmin=orig_np.min(), vmax=orig_np.max())
axes[1].set_title(f'ELM-INR  PSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}', fontsize=12)
axes[1].axis('off')

err = np.abs(orig_np - recon_np)
im = axes[2].imshow(err, cmap='hot', vmin=0, vmax=err.max())
axes[2].set_title(f'|Error|  max={err.max():.4f}', fontsize=12)
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.suptitle('ELM-INR: Single-Frame Reconstruction', fontsize=14)
plt.tight_layout()
plt.show()
torch.cuda.empty_cache()

## 6. Compression Ratio Analysis

The ELM-INR stores:
- **Learned**: output weights $\alpha_i$ per subdomain  ($K \times m \times 1$ floats)
- **Shared / regenerable**: RFF matrix $B$ and hidden weights $W_h, b_h$
  (deterministic from seed — only the seed is needed)

Raw image: $H \times W \times 2$ bytes (int16) or $H \times W \times 4$ bytes (float32).

In [ ]:
storage = model.storage_bytes(dtype_bytes=4)  # float32
raw_int16 = H * W * 2
raw_f32 = H * W * 4

print("Storage breakdown (float32):")
for k, v in storage.items():
    print(f"  {k:>30s}: {v:>12,} bytes  ({v/1e6:.3f} MB)")

print(f"\nRaw image sizes:")
print(f"  int16 : {raw_int16:>12,} bytes  ({raw_int16/1e6:.3f} MB)")
print(f"  float32: {raw_f32:>12,} bytes  ({raw_f32/1e6:.3f} MB)")

cr_stored = raw_int16 / storage['total_stored']
cr_all = raw_int16 / storage['total_if_all_stored']

print(f"\nCompression ratio (vs int16 raw):")
print(f"  Seed-regenerable (store alpha + B + meta): {cr_stored:.2f}x")
print(f"  Store everything (alpha + B + W_h + b_h) : {cr_all:.4f}x")

## 7. Adaptive Refinement

**Idea:** Start from the uniform-grid ELM-INR, then iteratively:
1. Compute the per-pixel residual error $|I(x,y) - \hat{f}(x,y)|$.
2. Greedily place $\lfloor \alpha \cdot N_{\text{sub}} \rfloor$ new subdomains centred on the highest-error pixels (with minimum spacing to diversify placement).
3. Each new subdomain gets fresh random hidden weights and is fitted in closed form to the original image within its support.
4. The partition of unity automatically adjusts: `predict()` normalises over **all** subdomains (original + new), so the new windows blend in naturally.
5. Repeat until `max_iters` or a target PSNR / SSIM is reached.

Because every local ELM is fitted independently to the same target image, old subdomains' weights $\alpha_i$ remain optimal — we only solve for the **new** subdomains. The quality boost comes from the **ensemble effect**: each new subdomain uses different random features, reducing variance in high-error regions.

In [ ]:
@dataclass
class AdaptiveRefinementConfig:
    """Parameters for the iterative adaptive refinement loop."""
    alpha: float = 0.1              # fraction of current subdomain count to add per iter
    max_iters: int = 10             # max refinement iterations
    target_psnr: float = 50.0       # stop early if PSNR >= this
    target_ssim: float = 0.995      # stop early if SSIM >= this
    min_spacing_pixels: int = 5     # minimum pixel distance between new subdomain centres
    subdomain_scale: float = 1.0    # size of new subdomains relative to base grid cell
    min_hidden_dim: int = 16        # floor for area-scaled hidden_dim of adaptive subdomains


# ── Helpers ────────────────────────────────────────────────────────────────

def _greedy_select_centers(error_map: np.ndarray, n: int,
                           min_spacing: int) -> list:
    """
    Greedily select *n* pixel locations with highest absolute error,
    enforcing a minimum Chebyshev distance between selections.

    Returns a list of (row, col) tuples.
    """
    H, W = error_map.shape
    err = error_map.copy()
    centers = []

    for _ in range(n):
        idx = int(np.argmax(err))
        r, c = divmod(idx, W)
        if err[r, c] <= 0:
            break  # no more positive-error pixels
        centers.append((r, c))

        # Zero out a neighbourhood to enforce spacing
        r_lo, r_hi = max(0, r - min_spacing), min(H, r + min_spacing + 1)
        c_lo, c_hi = max(0, c - min_spacing), min(W, c + min_spacing + 1)
        err[r_lo:r_hi, c_lo:c_hi] = 0.0

    return centers


def _pixel_to_norm(row: int, col: int, H: int, W: int) -> tuple:
    """Convert pixel (row, col) to normalised [-1, 1] coordinates."""
    cy = -1.0 + 2.0 * (row + 0.5) / H
    cx = -1.0 + 2.0 * (col + 0.5) / W
    return cy, cx


def _make_subdomain(model: ELMINR2D, cy: float, cx: float,
                    hy: float, hx: float, hidden_dim: int) -> dict:
    """
    Create a new subdomain dict with fresh random hidden weights
    drawn from *model.rng* (deterministic continuation of the stream).

    hidden_dim is passed explicitly so adaptive subdomains can be smaller
    than the base grid subdomains (scaled proportional to subdomain area).
    """
    cfg = model.cfg
    if cfg.expand_dim > 0:
        std1 = 1.0 / math.sqrt(model.input_dim)
        W_h1 = torch.randn(model.input_dim, cfg.expand_dim,
                            generator=model.rng, device=model.device) * std1
        b_h1 = torch.rand(cfg.expand_dim,
                           generator=model.rng, device=model.device)
        std2 = 1.0 / math.sqrt(cfg.expand_dim)
        W_h2 = torch.randn(cfg.expand_dim, hidden_dim,
                            generator=model.rng, device=model.device) * std2
        b_h2 = torch.rand(hidden_dim,
                           generator=model.rng, device=model.device)
        return {
            'center': (cy, cx), 'half_size': (hy, hx),
            'W_h1': W_h1, 'b_h1': b_h1,
            'W_h2': W_h2, 'b_h2': b_h2,
        }
    else:
        std = 1.0 / math.sqrt(model.input_dim)
        W_h = torch.randn(model.input_dim, hidden_dim,
                           generator=model.rng, device=model.device) * std
        b_h = torch.rand(hidden_dim,
                          generator=model.rng, device=model.device)
        return {
            'center': (cy, cx), 'half_size': (hy, hx),
            'W_h': W_h, 'b_h': b_h,
        }


def _fit_subdomain(model: ELMINR2D, sub: dict,
                   coords: torch.Tensor, features: torch.Tensor,
                   Y: torch.Tensor) -> torch.Tensor:
    """
    Fit a single subdomain's output weights in closed form.
    Returns alpha  [hidden_dim, 1].
    hidden_dim is read from the subdomain's own weight shape so it works
    for both full-size base subdomains and smaller adaptive ones.
    """
    cy, cx = sub['center']
    hy, hx = sub['half_size']
    mask = ((coords[:, 0] - cy).abs() <= hy) & \
           ((coords[:, 1] - cx).abs() <= hx)
    n_pts = mask.sum().item()

    # Derive actual hidden_dim from stored weights (handles variable sizes)
    actual_hidden_dim = (sub['W_h2'] if 'W_h2' in sub else sub['W_h']).shape[1]

    if n_pts == 0:
        return torch.zeros(actual_hidden_dim, 1, device=model.device)

    H_sub = model._hidden_matrix(features[mask], sub)
    return model._solve_alpha(H_sub, Y[mask])


print("Adaptive helpers ready")

In [ ]:
def adaptive_refinement(
    image: torch.Tensor,
    base_cfg: ELMINRConfig,
    adapt_cfg: AdaptiveRefinementConfig,
    device: torch.device,
    verbose: bool = True,
):
    """
    Iterative adaptive refinement of ELM-INR.

    Returns
    -------
    model : ELMINR2D
        The model with all subdomains (uniform + adaptive).
    history : list[dict]
        Per-iteration metrics (psnr, ssim, n_subdomains, cr, error_map).
    """
    Himg, Wimg = image.shape
    orig_np = image.cpu().numpy()

    # ── 1. Initial uniform-grid fit ──────────────────────────────────────
    model = ELMINR2D(base_cfg, device)
    model.fit(image, verbose=verbose)

    # Base half-sizes used for new subdomains
    base_hy = (1.0 / base_cfg.grid_h) * (1.0 + base_cfg.overlap) * adapt_cfg.subdomain_scale
    base_hx = (1.0 / base_cfg.grid_w) * (1.0 + base_cfg.overlap) * adapt_cfg.subdomain_scale

    # hidden_dim scaled proportional to subdomain area (scale²), with a floor
    adaptive_hidden_dim = max(
        adapt_cfg.min_hidden_dim,
        int(base_cfg.hidden_dim * adapt_cfg.subdomain_scale ** 2),
    )
    if verbose:
        print(f"  Adaptive subdomain hidden_dim = {adaptive_hidden_dim} "
              f"(base={base_cfg.hidden_dim}, scale²={adapt_cfg.subdomain_scale**2:.2f})")

    # Precompute shared coordinate / feature tensors
    coords   = model.make_coords(Himg, Wimg, device)
    features = model.encode(coords)
    Y        = image.flatten().unsqueeze(-1)

    history = []
    t_start = time.time()

    for iteration in range(adapt_cfg.max_iters + 1):
        # ── Evaluate current reconstruction ──────────────────────────────
        recon    = model.reconstruct_image()
        recon_np = recon.clamp(0, 1).cpu().numpy()
        err_map  = np.abs(orig_np - recon_np)

        psnr = compute_psnr(recon_np, orig_np)
        ssim = compute_ssim(recon_np, orig_np)
        n_sub = len(model.subdomains)

        # Compression ratio: raw int16 bytes vs stored (alpha + B + meta)
        storage = model.storage_bytes(dtype_bytes=4)
        raw_int16 = Himg * Wimg * 2
        cr = raw_int16 / storage['total_stored']

        history.append({
            'iter':         iteration,
            'psnr':         psnr,
            'ssim':         ssim,
            'n_subdomains': n_sub,
            'cr':           cr,
            'error_map':    err_map.copy() if iteration <= 5 else None,
            'elapsed':      time.time() - t_start,
        })

        if verbose:
            print(f"[Iter {iteration:>2d}]  PSNR = {psnr:.2f} dB  |  "
                  f"SSIM = {ssim:.4f}  |  CR = {cr:.2f}x  |  subdomains = {n_sub}")

        # ── Stopping criteria ────────────────────────────────────────────
        if iteration > 0 and (psnr >= adapt_cfg.target_psnr
                              or ssim >= adapt_cfg.target_ssim):
            if verbose:
                print(f"  >> Target reached — stopping.")
            break
        if iteration == adapt_cfg.max_iters:
            break

        # ── Greedy centre selection ──────────────────────────────────────
        n_new = max(1, int(adapt_cfg.alpha * n_sub))
        new_centres = _greedy_select_centers(
            err_map, n_new, adapt_cfg.min_spacing_pixels,
        )
        if verbose:
            top_err = err_map.max()
            print(f"  Adding {len(new_centres)} subdomains  "
                  f"(max |err| = {top_err:.5f})")

        # ── Create & fit new subdomains ──────────────────────────────────
        for r, c in new_centres:
            cy, cx = _pixel_to_norm(r, c, Himg, Wimg)
            sub = _make_subdomain(model, cy, cx, base_hy, base_hx,
                                  hidden_dim=adaptive_hidden_dim)
            alpha = _fit_subdomain(model, sub, coords, features, Y)
            model.subdomains.append(sub)
            model.alphas.append(alpha)

        torch.cuda.empty_cache()

    elapsed = time.time() - t_start
    if verbose:
        print(f"\nAdaptive refinement finished in {elapsed:.1f}s  "
              f"({len(model.subdomains)} total subdomains)")

    return model, history

### Run Adaptive Refinement

In [ ]:
# ── Base ELM-INR config (same as Section 4) ──────────────────────────────
base_cfg = ELMINRConfig(
    grid_h=32,
    grid_w=32,
    overlap=0.2,
    hidden_dim=512,
    expand_dim=0,
    activation='relu',
    use_rff=True,
    rff_dim=512,
    rff_sigma=5.0,
    ridge_lambda=1e-8,
    window='gaussian',
    seed=42,
)

# ── Adaptive refinement config ───────────────────────────────────────────
adapt_cfg = AdaptiveRefinementConfig(
    alpha=0.05,                # add 5% more subdomains per iteration
    max_iters=50,
    target_psnr=55.0,
    target_ssim=0.998,
    min_spacing_pixels=5,      # allow fairly dense placement
    subdomain_scale=0.05,       # 1.0 means same size as base grid cells
)

# ── Run ──────────────────────────────────────────────────────────────────
image_adapt = frames[FRAME_IDX].to(device)
adapt_model, adapt_history = adaptive_refinement(
    image_adapt, base_cfg, adapt_cfg, device, verbose=True,
)
torch.cuda.empty_cache()

In [ ]:
# ── Convergence curves ────────────────────────────────────────────────────
iters   = [h['iter'] for h in adapt_history]
psnrs   = [h['psnr'] for h in adapt_history]
ssims   = [h['ssim'] for h in adapt_history]
n_subs  = [h['n_subdomains'] for h in adapt_history]
crs     = [h['cr'] for h in adapt_history]

fig, axes = plt.subplots(1, 4, figsize=(22, 4))

axes[0].plot(iters, psnrs, 'o-', color='tab:blue')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('PSNR vs Iteration')
axes[0].grid(True, alpha=0.3)

axes[1].plot(iters, ssims, 's-', color='tab:green')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('SSIM')
axes[1].set_title('SSIM vs Iteration')
axes[1].grid(True, alpha=0.3)

axes[2].plot(iters, crs, '^-', color='tab:red')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Compression Ratio (×)')
axes[2].set_title('CR vs Iteration  (raw int16 / stored)')
axes[2].grid(True, alpha=0.3)
# annotate first and last CR values
for _it, _cr in [(iters[0], crs[0]), (iters[-1], crs[-1])]:
    axes[2].annotate(f'{_cr:.2f}×', xy=(_it, _cr),
                     xytext=(4, 4), textcoords='offset points', fontsize=8)

axes[3].plot(iters, n_subs, 'D-', color='tab:orange')
axes[3].set_xlabel('Iteration')
axes[3].set_ylabel('Total Subdomains')
axes[3].set_title('Subdomain Count vs Iteration')
axes[3].grid(True, alpha=0.3)

plt.suptitle('Adaptive Refinement Convergence', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nCompression ratio:  {crs[0]:.2f}× (uniform)  →  {crs[-1]:.2f}×"
      f" (after {iters[-1]} adaptive iter{'s' if iters[-1] != 1 else ''})")
print(f"Quality gain:       PSNR {psnrs[0]:.2f} → {psnrs[-1]:.2f} dB  |"
      f"  SSIM {ssims[0]:.4f} → {ssims[-1]:.4f}")

In [ ]:
# ── Error map evolution ───────────────────────────────────────────────────
err_maps = [(h['iter'], h['error_map']) for h in adapt_history
            if h['error_map'] is not None]

n_maps = len(err_maps)
fig, axes = plt.subplots(1, n_maps, figsize=(4 * n_maps, 4))
if n_maps == 1:
    axes = [axes]

# Shared colour scale from iteration 0
vmax = err_maps[0][1].max()

for ax, (it, emap) in zip(axes, f):
    im = ax.imshow(emap, cmap='hot', vmin=0, vmax=vmax)
    ax.set_title(f'Iter {it}  (max={emap.max():.4f})')
    ax.axis('off')

plt.suptitle('|Error| maps across adaptive iterations', fontsize=13)
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final before / after comparison ───────────────────────────────────────
adapt_recon    = adapt_model.reconstruct_image()
adapt_recon_np = adapt_recon.clamp(0, 1).cpu().numpy()
adapt_orig_np  = image_adapt.cpu().numpy()

final_psnr = compute_psnr(adapt_recon_np, adapt_orig_np)
final_ssim = compute_ssim(adapt_recon_np, adapt_orig_np)

# Uniform baseline (from Section 5)
base_psnr = adapt_history[0]['psnr']
base_ssim = adapt_history[0]['ssim']

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].imshow(adapt_orig_np, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Original (frame {FRAME_IDX})')
axes[0].axis('off')

# Uniform-grid reconstruction (iteration 0 error map as proxy)
axes[1].imshow(adapt_orig_np - adapt_history[0]['error_map'],
               cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Uniform grid\nPSNR={base_psnr:.2f}  SSIM={base_ssim:.4f}')
axes[1].axis('off')

axes[2].imshow(adapt_recon_np, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Adaptive ({adapt_history[-1]["iter"]} iters)\n'
                  f'PSNR={final_psnr:.2f}  SSIM={final_ssim:.4f}')
axes[2].axis('off')

err_final = np.abs(adapt_orig_np - adapt_recon_np)
im = axes[3].imshow(err_final, cmap='hot', vmin=0,
                    vmax=adapt_history[0]['error_map'].max())
axes[3].set_title(f'|Error| after adaptive\nmax={err_final.max():.5f}')
axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046)

plt.suptitle(f'Adaptive Refinement:  {adapt_history[0]["n_subdomains"]} → '
             f'{adapt_history[-1]["n_subdomains"]} subdomains  |  '
             f'PSNR {base_psnr:.2f} → {final_psnr:.2f} dB  |  '
             f'SSIM {base_ssim:.4f} → {final_ssim:.4f}',
             fontsize=13)
plt.tight_layout()
plt.show()

# ── Subdomain placement scatter ─────────────────────────────────────────
n_base = base_cfg.grid_h * base_cfg.grid_w
base_centers = [s['center'] for s in adapt_model.subdomains[:n_base]]
new_centers  = [s['center'] for s in adapt_model.subdomains[n_base:]]

fig, ax = plt.subplots(figsize=(8, 7))
if base_centers:
    by, bx = zip(*base_centers)
    ax.scatter(bx, by, s=2, c='steelblue', alpha=0.3, label=f'Uniform ({n_base})')
if new_centers:
    ny, nx = zip(*new_centers)
    ax.scatter(nx, ny, s=12, c='red', alpha=0.7, zorder=5,
               label=f'Adaptive ({len(new_centers)})')
ax.set_xlim(-1, 1); ax.set_ylim(1, -1)  # flip y to match image
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Subdomain centres (normalised coords)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

torch.cuda.empty_cache()